# GPU-Accelerated Data Processing Pipeline
### NVIDIA — Fundamentals of Accelerated Computing with CUDA Python (Course)

Three GPU optimization techniques from the course, each benchmarked against a CPU baseline. This is bascilly a way for me to put everything I have learned in that course into practice so I can really understand High Performance Computing.

In [1]:
import time
import math

In [2]:
n = 4096*4096
threads_per_block = 256 #256/32 = 8 warps per block
blocks = int(n / threads_per_block)
grid_size = 256 # 256*256 = 65536 threads in the grid

### Technique 1: Grid Stride Loop

**A grid stride loop launches a fixed grid and lets each thread stride across the full array, keeping memory coalesced**

Why use this technique?
  * Same kernel works for any size array
  * Adjacent threads always read adjacent memory addresses

In [3]:
import numpy as np
from numba import jit, cuda

#cpu normalization first

@jit(nopython=True)
def cpu_normalization(arr, min_val, max_val):

  out = np.empty_like(arr)
  range_val = max_val - min_val

  for i in range(len(arr)):
    out[i] = (arr[i] - min_val) / range_val

  return out

#gpu normalization naive (1 thread per element)

@cuda.jit
def gpu_normalization_naive(arr, out, min_val, range):

  idx = cuda.grid(1)

  if idx < arr.shape[0]:
    out[idx] = (arr[idx] - min_val) / range

#gpu normalization stride
@cuda.jit
def gpu_normalization_stride(arr, out, min_val, range):

  start = cuda.grid(1)
  stride = cuda.gridsize(1)
  n = arr.shape[0]
  i = start

  while i < n:
    out[i] = (arr[i] - min_val) / range
    i += stride


## Data and GPU Setup

In [4]:
range_filler = np.random.default_rng(42)
data = {"values": (range_filler.random(n) * 1000).astype(np.float64)}

arr = data["values"].astype(np.float32)
min_val = float(arr.min())
max_val = float(arr.max())
range_val = max_val - min_val

cpu_normalization(arr[:256], min_val, max_val) #trigger JIT compile before timing

d_arr = cuda.to_device(arr) #CPU RAM to GPU VRAM
d_out = cuda.device_array_like(arr) #allocate empty output on GPU

naive_blocks = math.ceil(len(arr) / threads_per_block)


## Checking Benchmarks

In [5]:
%timeit cpu_normalization(arr, min_val, max_val)

48.3 ms ± 451 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
%time gpu_normalization_naive[naive_blocks, threads_per_block](d_arr, d_out, min_val, range_val); cuda.synchronize()

CPU times: user 696 ms, sys: 66.7 ms, total: 762 ms
Wall time: 1.4 s


In [7]:
%time gpu_normalization_stride[grid_size, threads_per_block](d_arr, d_out, min_val, range_val); cuda.synchronize()

CPU times: user 113 ms, sys: 514 µs, total: 113 ms
Wall time: 115 ms


In [8]:
%timeit gpu_normalization_stride[grid_size, threads_per_block](d_arr, d_out, min_val, range_val); cuda.synchronize()

1.88 ms ± 16.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Technique 2: Shared Memory

**Shared memory is a small, fast on-chip memory (~100x lower latency than global GPU memory) shared between all threads in the same block**

Why use this technique?
  * The naive transpose writes to `transposed[x][y] = a[y][x]` which reads are coalesced but writes jump around in memory (uncoalesced) which is slow
  * Loading a 32x32 tile into shared memory first fixes this: threads read the tile into fast on-chip memory, `cuda.syncthreads()` waits for all threads to finish loading, then writes back to global memory at the transposed location

In [9]:
from numba import types as numba_types

#matrix transpose naive (coalesced reads, uncoalesced writes)

@cuda.jit
def gpu_transpose_naive(a, transposed):

  x, y = cuda.grid(2) #2D thread coordinates

  if x < a.shape[0] and y < a.shape[1]:
    transposed[x][y] = a[y][x] #write jumps around in memory

#matrix transpose shared memory (both reads and writes coalesced)

@cuda.jit
def gpu_transpose_shared(a, transposed):

  #1) allocate 32x32 shared memory tile — shape must be a compile-time constant
  tile = cuda.shared.array((32, 32), numba_types.float32)

  #input coordinates (where to READ from in a)
  x = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x #column
  y = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y #row

  #2) coalesced read: global memory -> shared memory
  if x < a.shape[1] and y < a.shape[0]:
    tile[cuda.threadIdx.y, cuda.threadIdx.x] = a[y, x]

  #3) barrier: all threads must finish loading before anyone reads
  cuda.syncthreads()

  #output coordinates (blocks swapped for transpose)
  t_x = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.x
  t_y = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.y

  #4) coalesced write: shared memory -> global memory
  if t_x < transposed.shape[1] and t_y < transposed.shape[0]:
    transposed[t_y, t_x] = tile[cuda.threadIdx.x, cuda.threadIdx.y]


## Data and GPU Setup

In [10]:
matrix_n = 4096
n_elements = matrix_n * matrix_n

a          = np.arange(n_elements).reshape((matrix_n, matrix_n)).astype(np.float32)
transposed = np.zeros_like(a)

d_a          = cuda.to_device(a)          #CPU RAM to GPU VRAM
d_transposed = cuda.to_device(transposed) #CPU RAM to GPU VRAM

#same 2D grid config as Section 3 assessment
#32x32 block = 1024 threads (hardware max per block)
#128x128 grid covers the full 4096x4096 matrix
threads_2d = (32, 32)
blocks_2d  = (matrix_n // 32, matrix_n // 32) #(128, 128)


## Checking Benchmarks

In [11]:
%timeit np.transpose(a)

812 ns ± 225 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [12]:
%timeit gpu_transpose_naive[blocks_2d, threads_2d](d_a, d_transposed); cuda.synchronize()

1.69 ms ± 10.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
%timeit gpu_transpose_shared[blocks_2d, threads_2d](d_a, d_transposed); cuda.synchronize()

1.06 ms ± 10.4 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [14]:
#correctness check — same pattern used in your Section 3 assessment
result   = d_transposed.copy_to_host() #GPU VRAM to CPU RAM
expected = a.T
np.array_equal(result, expected)

True

### Technique 3: Atomic Operations

**`cuda.atomic.add` performs read → add → write as one indivisible hardware operation — no other thread can interrupt it**

Why use this technique?
  * Thousands of threads run simultaneously. Without atomics, two threads hitting the same bin at the same time cause a race condition:

```
Thread A reads  bin[5] = 10
Thread B reads  bin[5] = 10   # reads BEFORE A writes back
Thread A writes bin[5] = 11
Thread B writes bin[5] = 11   # overwrites A — one +1 is lost
Correct answer: 12   Got: 11
```

  * The broken kernel below shows the total count will be less than n

In [15]:
#cpu histogram baseline

def cpu_histogram(x, x_min, x_max, n_bins):
  counts, _ = np.histogram(x, bins=n_bins, range=(x_min, x_max))
  return counts.astype(np.int32)

#gpu histogram with race condition (intentionally broken)

@cuda.jit
def gpu_histogram_race(x, x_min, x_max, histogram_out):

  n_bins    = histogram_out.shape[0]
  bin_width = (x_max - x_min) / n_bins
  start     = cuda.grid(1)
  stride    = cuda.gridsize(1) #grid stride loop
  i = start

  while i < x.shape[0]:
    b = int((x[i] - x_min) / bin_width)
    if 0 <= b < n_bins:
      histogram_out[b] += 1 #RACE CONDITION
    i += stride

#gpu histogram with atomic operations (correct)

@cuda.jit
def gpu_histogram_atomic(x, x_min, x_max, histogram_out):

  n_bins    = histogram_out.shape[0]
  bin_width = (x_max - x_min) / n_bins
  start     = cuda.grid(1)
  stride    = cuda.gridsize(1) #grid stride loop
  i = start

  while i < x.shape[0]:
    b = int((x[i] - x_min) / bin_width)
    if 0 <= b < n_bins:
      cuda.atomic.add(histogram_out, b, 1) #thread-safe increment
    i += stride


## Data and GPU Setup

In [16]:
n_bins = 64
x      = data["values"].astype(np.float32)
x_min  = np.float32(x.min())
x_max  = np.float32(x.max())

d_x       = cuda.to_device(x)                              #CPU RAM to GPU VRAM
d_race    = cuda.to_device(np.zeros(n_bins, dtype=np.int32))
d_atomic  = cuda.to_device(np.zeros(n_bins, dtype=np.int32))

cpu_hist = cpu_histogram(x, x_min, x_max, n_bins)
print(f'CPU histogram total: {cpu_hist.sum():,}  (should equal n)')


CPU histogram total: 16,777,216  (should equal n)


## Checking Benchmarks

In [17]:
%timeit cpu_histogram(x, x_min, x_max, n_bins)

226 ms ± 49.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
#broken — race condition, total will be less than n
%time gpu_histogram_race[grid_size, threads_per_block](d_x, x_min, x_max, d_race); cuda.synchronize()
race_result = d_race.copy_to_host()
print(f'Race total:   {race_result.sum():,}  (expected {len(x):,}) <- lost writes!')


CPU times: user 146 ms, sys: 1.01 ms, total: 147 ms
Wall time: 147 ms
Race total:   143,854  (expected 16,777,216) <- lost writes!


In [19]:
#correct — atomic ops
%timeit gpu_histogram_atomic[grid_size, threads_per_block](d_x, x_min, x_max, d_atomic); cuda.synchronize()

6.1 ms ± 2.95 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
#correctness check
atomic_result = d_atomic.copy_to_host() #GPU VRAM to CPU RAM
print(f'Atomic total: {atomic_result.sum():,}  (expected {len(x):,})')
np.allclose(atomic_result, cpu_hist, atol=2)


Atomic total: 13,606,321,365  (expected 16,777,216)


False

---
### How to apply these to other projects

These three techniques cover the majority of real GPU workloads:

- **Grid stride loop** — use this by default on any 1D operation over a large array. If you're normalizing, scaling, transforming, or applying a formula element-wise to millions of rows, use this pattern. It works for any size dataset without changing the launch config.

- **Shared memory** — use this whenever threads in the same block need to read overlapping data. Common examples: image convolution (each pixel needs its neighbours), stencil computations, any sliding window operation. The tile size is almost always 32×32 for 2D problems.

- **Atomic operations** — use this whenever many threads write to the same output location. Common examples: counting, voting, any kind of binning or aggregation. If you find yourself doing `output[some_index] += value` inside a kernel, it needs `cuda.atomic.add`.
